In [ ]:
#@title ⚙️ Setup — choose how to provide the inputs (RUN ME FIRST)
#@markdown Pick how the input files for this step are provided:
#@markdown - **Mobile mode**: the files are downloaded automatically from the GitHub repo — no manual upload, works on phones/tablets.
#@markdown - **Manual upload**: you upload the files yourself in the cells below.
input_mode = "Mobile mode (auto-download from repo)" #@param ["Mobile mode (auto-download from repo)", "Manual upload"]

import os, urllib.request
REPO_RAW = "https://raw.githubusercontent.com/biochorl/Nanobody_de_novo_design/main"

# Inputs needed by THIS step.  (some links are FACSIMILE placeholders until the files
# are added to the repo — they will start working once they are committed)
INPUT_FILES = [('/content/IgGM_input.fasta', '/Example_output/IgGM_input.fasta', 'facsimile'), ('/content/antigen_A.pdb', '/Example_output/antigen_A.pdb', 'facsimile')]

def fetch_inputs(files):
    ok = 0
    for dst, rel, kind in files:
        os.makedirs(os.path.dirname(dst), exist_ok=True)
        url = REPO_RAW + rel
        try:
            urllib.request.urlretrieve(url, dst)
            print(f"✅ {os.path.basename(dst):<28} ← {url}  [{kind}]")
            ok += 1
        except Exception as e:
            tag = "facsimile not in repo yet" if kind == "facsimile" else "ERROR"
            print(f"⚠️  {os.path.basename(dst):<28} could not be downloaded ({tag}): {e}")
    return ok

if input_mode.startswith("Mobile"):
    print("📱 Mobile mode — downloading this step's inputs from the repo:\n")
    n = fetch_inputs(INPUT_FILES)
    print(f"\n→ {n}/{len(INPUT_FILES)} file(s) ready in /content. "
          "Skip the manual-upload cells and point the path fields to these files.")
else:
    print("🖱️ Manual mode — upload your files in the cells below as usual.")


# Step 2 — All-atom *de novo* nanobody design with IgGM

[IgGM](https://github.com/TencentAI4S/IgGM) co-designs the **sequence and structure** of the nanobody CDRs against your antigen and writes a nanobody–antigen complex in a single pass.

**Inputs (from Step 1, point 3):**
- `IgGM_input.fasta` — `>H` nanobody with the CDRs (redesign regions) as `X`, then `>A` antigen (antigen **last**)
- `antigen_A.pdb` — single-chain antigen
- the target epitope as **0-based positional indices** printed by the DiscoTope cell

**Output:** Designed `*.pdb` complexes → feed them to the **PIPPack** step for well-formed, repacked all-atoms side chains.

> ⚙️ Set the runtime to **GPU** (Runtime → Change runtime type → T4 GPU). The first design run downloads the IgGM model weights once (Zenodo).

In [ ]:
#@title Verify the runtime has a GPU
!nvidia-smi -L || echo "⚠️ No GPU detected. Go to Runtime > Change runtime type > T4 GPU, then re-run."


# Install IgGM (minimal)

IgGM imports only a handful of packages — most are already in Colab. We **do not** reinstall numpy / scipy / torch (mixing versions is what caused the earlier `scipy _spropack` error) and we **skip torch-geometric / ANARCI / PyRosetta / condacolab** because the design path never imports them. No session restart is needed.

In [ ]:
#@title Clone the official IgGM repository
!git clone -q https://github.com/TencentAI4S/IgGM /content/IgGM && echo "✅ IgGM cloned to /content/IgGM"


In [ ]:
#@title Install IgGM dependencies (minimal — the only packages IgGM actually needs)
#@markdown Already-in-Colab: torch, numpy, scipy, tqdm, pyyaml, requests. Installed here: biopython, ml-collections, dm-tree, openmm, pdbfixer (all-atom completion) + py3Dmol (visualization).
%pip install -q biopython ml-collections dm-tree openmm pdbfixer py3Dmol termcolor
import importlib
ok = []
for mod in ["Bio", "ml_collections", "tree", "openmm", "pdbfixer", "py3Dmol"]:
    try:
        importlib.import_module(mod); ok.append(mod)
    except Exception as e:
        print(f"❌ {mod}: {e}")
print("✅ Imported OK:", ", ".join(ok))
print("ℹ️ No session restart needed — continue to 'Run the design'.")


# Run the design

In [ ]:
#@title 1. Upload the Step 1 outputs (IgGM_input.fasta + antigen_A.pdb)
from google.colab import files
import os
print("Upload BOTH files saved in Step 1 (point 3): IgGM_input.fasta and antigen_A.pdb")
up = files.upload()
for fn in up:
    dst = "/content/" + os.path.basename(fn)
    with open(dst, "wb") as f:
        f.write(up[fn])
    print("saved ->", dst)
present = [f for f in os.listdir('/content') if f.endswith(('.fasta', '.pdb'))]
print("\nInput files now in /content:", present)


In [ ]:
#@title 2. Generate all-atom nanobody designs with IgGM 🧬
#@markdown IgGM co-designs the **sequence + structure** of the nanobody CDRs (the `X` regions) against the antigen and writes a
#@markdown **full all-atom** nanobody–antigen complex (side chains completed via PDBFixer; optional Amber/Rosetta relax).
#@markdown This single step replaces the old **RFdiffusion (backbone)** + **ProteinMPNN (sequence)** stages.
import os, re, sys, subprocess, random

#@markdown ## 1. Core inputs (from Step 1)
#@markdown **--fasta:** combined FASTA — `>H` nanobody (CDRs = `X`) + `>A` antigen (antigen last).
fasta_path = "/content/IgGM_input.fasta" #@param {type:"string"}
#@markdown **--antigen:** single-chain antigen PDB (`antigen_A.pdb` from Step 1).
antigen_pdb_path = "/content/antigen_A.pdb" #@param {type:"string"}

#@markdown ## 2. Epitope
#@markdown **--epitope:** space-separated **0-based positional indices** printed by the Step 1 DiscoTope cell (paste one hotspot). Leave empty only if your antigen PDB is a real complex.
epitope_residues = "" #@param {type:"string"}

#@markdown ## 3. Sampling
#@markdown **--num_samples:** number of designs to generate. With CDR3-length variation ON (below), each design uses an independently sampled CDR3 length.
num_samples = 5 #@param {type:"number"}
#@markdown **--steps:** sampling steps (10 is a good start; higher = slower/better).
steps = 10 #@param {type:"number"}

#@markdown ## CDR3 length exploration (nanobody)
#@markdown If ON, each of the `num_samples` designs gets a CDR3 length sampled uniformly in `[cdr3_min, cdr3_max]` (typical nanobody CDR3 range). The CDR3 is taken as the **last block of `X`** in the `>H` chain. One design is generated per sampled length.
vary_cdr3_length = True #@param {type:"boolean"}
cdr3_min = 7 #@param {type:"integer"}
cdr3_max = 20 #@param {type:"integer"}
length_seed = 42 #@param {type:"integer"}

#@markdown ## 4. Advanced & output
#@markdown **--chunk_size:** lower only if you hit out-of-memory errors.
chunk_size = 64 #@param {type:"number"}
#@markdown **--max_antigen_size:** large antigens are cropped around the epitope to save memory.
max_antigen_size = 2000 #@param {type:"number"}
#@markdown **--relax:** Amber/Rosetta relaxation (needs PyRosetta). Leave OFF — the next step (PIPPack) repacks side chains.
relax = False #@param {type:"boolean"}
output_directory = "/content/IgGM/outputs/designs" #@param {type:"string"}

script_path = "/content/IgGM/design.py"

def read_fasta(path):
    recs, h, buf = [], None, []
    for line in open(path):
        line = line.strip()
        if not line:
            continue
        if line.startswith('>'):
            if h is not None:
                recs.append((h, ''.join(buf)))
            h, buf = line[1:].strip(), []
        else:
            buf.append(line)
    if h is not None:
        recs.append((h, ''.join(buf)))
    return recs

def run_design(fasta, n_samples):
    """Run IgGM design.py on one FASTA, streaming output. Returns the exit code."""
    command = [
        "python", script_path,
        "--fasta", fasta,
        "--antigen", antigen_pdb_path,
        "--output", output_directory,
        "--num_samples", str(n_samples),
        "--steps", str(steps),
        "--chunk_size", str(chunk_size),
        "--max_antigen_size", str(max_antigen_size),
    ]
    if epitope_residues.strip():
        command.append("--epitope")
        command.extend(epitope_residues.strip().split())
    if relax:
        command.append("--relax")
    print("▶️ " + " ".join(command))
    proc = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                            text=True, bufsize=1, cwd="/content/IgGM")
    for line in iter(proc.stdout.readline, ''):
        print(line, end=''); sys.stdout.flush()
    proc.stdout.close()
    return proc.wait()

if not os.path.exists(script_path):
    print(f"❌ design.py not found at {script_path}. Run the install cells first.")
elif not os.path.exists(fasta_path):
    print(f"❌ FASTA not found: {fasta_path}. Upload Step 1's IgGM_input.fasta.")
elif not os.path.exists(antigen_pdb_path):
    print(f"❌ Antigen PDB not found: {antigen_pdb_path}. Upload Step 1's antigen_A.pdb.")
else:
    os.makedirs(output_directory, exist_ok=True)
    records = read_fasta(fasta_path)

    # find the nanobody (>H or first non-antigen) and the antigen (last record)
    nb_idx = next((i for i, (hh, _) in enumerate(records) if hh.upper().startswith('H')), 0)
    nb_id, nb_seq = records[nb_idx]
    x_blocks = list(re.finditer(r'X+', nb_seq))

    if vary_cdr3_length and x_blocks:
        inputs_dir = "/content/IgGM/outputs/inputs"
        os.makedirs(inputs_dir, exist_ok=True)
        cdr3 = x_blocks[-1]                # CDR3 = last X block of the nanobody
        s, e = cdr3.start(), cdr3.end()
        rng = random.Random(length_seed)
        lo, hi = int(min(cdr3_min, cdr3_max)), int(max(cdr3_min, cdr3_max))
        print(f"CDR3 variation ON: native CDR3 length = {e - s}; sampling lengths in [{lo}, {hi}] "
              f"for {int(num_samples)} design(s).\n")

        failures = 0
        for i in range(int(num_samples)):
            L = rng.randint(lo, hi)
            nb_var = nb_seq[:s] + ('X' * L) + nb_seq[e:]
            variant_name = f"IgGM_input_cdr3-{L}_v{i}"
            variant_fasta = os.path.join(inputs_dir, f"{variant_name}.fasta")
            with open(variant_fasta, 'w') as f:
                for hh, seq in records:
                    seq_out = nb_var if hh == nb_id else seq
                    f.write(f">{hh}\n{seq_out}\n")
            print(f"=== Design {i+1}/{int(num_samples)} — CDR3 length {L} ===")
            rc = run_design(variant_fasta, 1)
            if rc != 0:
                failures += 1
                print(f"❌ design {i} (CDR3={L}) failed (exit {rc}).")
            print()
        print(f"✅ Done. {int(num_samples) - failures}/{int(num_samples)} designs in: {output_directory}")
        print("Output filenames encode the CDR3 length, e.g. IgGM_input_cdr3-14_v2_0.pdb")
    else:
        if vary_cdr3_length and not x_blocks:
            print("⚠️ No 'X' block found in the nanobody — running without CDR3 variation.")
        rc = run_design(fasta_path, int(num_samples))
        if rc == 0:
            print(f"\n✅ Done. All-atom designs saved in: {output_directory}")
        else:
            print(f"\n❌ IgGM failed (exit code {rc}).")

    # --- AUTOMATED CDR ANNOTATION GENERATION ---
    import json
    import glob

    annotations = {}
    if len(x_blocks) >= 3:
        cdr1_s, cdr1_e = x_blocks[0].start() + 1, x_blocks[0].end()
        cdr2_s, cdr2_e = x_blocks[1].start() + 1, x_blocks[1].end()

        for pdb_path in glob.glob(os.path.join(output_directory, "*.pdb")):
            base_name = os.path.basename(pdb_path)
            with open(pdb_path) as f:
                h_atoms = [l for l in f.read().splitlines() if l.startswith('ATOM') and l[21] == 'H']
                if h_atoms:
                    h_len = max(int(l[22:26].strip()) for l in h_atoms)
                    tail_len = len(nb_seq) - x_blocks[2].end()
                    cdr3_s, cdr3_e = x_blocks[2].start() + 1, h_len - tail_len

                    annotations[base_name] = {
                        "CDR1": [cdr1_s, cdr1_e],
                        "CDR2": [cdr2_s, cdr2_e],
                        "CDR3": [cdr3_s, cdr3_e]
                    }

        os.makedirs("/content/PIPPack_outputs", exist_ok=True)
        with open("/content/PIPPack_outputs/cdrs_annotation.json", "w") as f:
            json.dump(annotations, f, indent=4)
        print("\n    ✅ Saved CDR annotations for AntiFold to /content/PIPPack_outputs/cdrs_annotation.json")
    # -------------------------------------------

In [ ]:
#@title 3. Visualize an all-atom IgGM design
import os, glob
import py3Dmol
output_directory = "/content/IgGM/outputs/designs" #@param {type:"string"}
design_index = 0 #@param {type:"integer"}

designs = sorted(glob.glob(os.path.join(output_directory, "*.pdb")))
if not designs:
    print("No designs found. Run the IgGM design cell first.")
else:
    print("Available designs:")
    for i, d in enumerate(designs):
        print(f" [{i}] {os.path.basename(d)}")

    idx = min(max(0, design_index), len(designs)-1)
    selected_design = designs[idx]
    pdb_data = open(selected_design).read()

    n_atoms = sum(l.startswith('ATOM') for l in pdb_data.splitlines())
    n_side = sum(l.startswith('ATOM') and l[12:16].strip() not in ('N', 'CA', 'C', 'O')
                 for l in pdb_data.splitlines())
    print(f"\nShowing: {os.path.basename(selected_design)}")
    print(f"ATOM lines: {n_atoms} | side-chain atoms: {n_side} | all-atom: {n_side > 0}")

    view = py3Dmol.view(width=820, height=560)
    view.addModel(pdb_data, 'pdb')

    # Color Antigen (A) and Nanobody (H) differently
    view.setStyle({'chain': 'A'}, {'cartoon': {'color': 'orange'}})
    view.setStyle({'chain': 'H'}, {'cartoon': {'color': 'cyan'}})

    # Add transparent surfaces
    view.addSurface(py3Dmol.VDW, {'opacity': 0.4, 'color': 'orange'}, {'chain': 'A'})
    view.addSurface(py3Dmol.VDW, {'opacity': 0.4, 'color': 'cyan'}, {'chain': 'H'})

    # Highlight Nanobody sidechains at interface (optional, keep default sticks)
    view.addStyle({'chain': 'H'}, {'stick': {'radius': 0.15, 'color': 'cyan'}})
    view.addStyle({'chain': 'A'}, {'stick': {'radius': 0.15, 'color': 'orange'}})

    view.zoomTo()
    view.show()


In [ ]:
#@title 4. Install PIPPack
#@markdown This cell sets up PIPPack for side-chain refinement. It may take a minute or two.
!rm -rf /content/PIPPack
import os
if not os.path.exists("/content/PIPPack"):
  ! git clone https://github.com/Kuhlman-Lab/PIPPack.git /content/PIPPack
  ! pip uninstall -y wandb

! pip install omegaconf pytorch-lightning biopython nglview hydra-core torch_geometric


In [ ]:
#@title 5. Refine Side-chains with PIPPack
#@markdown This step refines the side-chains of the designs generated by IgGM.
import os, glob, sys, pickle
import sys
import pytorch_lightning
sys.modules['lightning'] = pytorch_lightning
import torch
import warnings
warnings.simplefilter('ignore')

sys.path.append('/content/PIPPack')
from data.protein import from_pdb_file
from data.top2018_dataset import transform_structure, collate_fn
import hydra
from utils.train_utils import load_checkpoint
from ensembled_inference import sample_epoch
from model.resampling import resample_loop
from inference import pdbs_from_prediction

input_directory = "/content/IgGM/outputs/designs"
output_directory = "/content/PIPPack_outputs"
os.makedirs(output_directory, exist_ok=True)

designs = sorted(glob.glob(os.path.join(input_directory, "*.pdb")))
if not designs:
    print("No designs found in", input_directory)
else:
    print(f"Found {len(designs)} designs to refine.")

    device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
    model_names = ["pippack_model_1", "pippack_model_2", "pippack_model_3"]

    # Parse one config to get n_chi_bins
    with open(f'/content/PIPPack/model_weights/{model_names[0]}_config.pickle', 'rb') as f:
        cfg = pickle.load(f)
    n_chi_bins = cfg.model.n_chi_bins

    print("Loading PIPPack ensemble...")
    models = []
    for model_name in model_names:
        cfg_file = f'/content/PIPPack/model_weights/{model_name}_config.pickle'
        ckpt_file = f'/content/PIPPack/model_weights/{model_name}_ckpt.pt'
        with open(cfg_file, 'rb') as f:
            cfg = pickle.load(f)
        m = hydra.utils.instantiate(cfg.model).to(device)
        load_checkpoint(ckpt_file, m)
        models.append(m)

    print("Running refinement...")
    from Bio.PDB import PDBParser, PDBIO, Select

    class SingleChainSelect(Select):
        def __init__(self, chain_id):
            self.chain_id = chain_id
        def accept_chain(self, chain):
            return chain.id == self.chain_id

    class ExcludeChainSelect(Select):
        def __init__(self, exclude_id):
            self.exclude_id = exclude_id
        def accept_chain(self, chain):
            return chain.id != self.exclude_id

    for pdb_file in designs:
        base_name = os.path.basename(pdb_file)
        print(f"\n--- Processing {base_name} ---")

        # 1. Split the complex into Nanobody and Antigen
        parser = PDBParser(QUIET=True)
        orig_struct = parser.get_structure("orig", pdb_file)
        io = PDBIO()
        io.set_structure(orig_struct)

        chain_order = [ch.id for ch in orig_struct[0]]
        if 'H' in chain_order:
            nb_id = 'H'
        else:
            nb_id = min(orig_struct[0], key=lambda c: len(list(c.get_residues()))).id

        print(f"[DEBUG] Original chains: {chain_order}. Selected Nanobody ID: {nb_id}")

        temp_h = "/content/temp_H.pdb"
        temp_a = "/content/temp_A.pdb"
        io.save(temp_h, SingleChainSelect(nb_id))
        io.save(temp_a, ExcludeChainSelect(nb_id))

        # --- Run PDBFixer to add missing Oxygen and sidechains ---
        import pdbfixer
        import openmm.app as app
        fixer = pdbfixer.PDBFixer(filename=temp_h)
        fixer.findMissingResidues()
        fixer.findMissingAtoms()
        fixer.addMissingAtoms()
        with open(temp_h, 'w') as f:
            app.PDBFile.writeFile(fixer.topology, fixer.positions, f, keepIds=True)
        # ---------------------------------------------------------

        with open(temp_h, 'r') as f:
            hlines = f.readlines()
        print(f"[DEBUG] Extracted Nanobody file has {len(hlines)} lines.")

        with open(temp_h, 'w') as f:
            for l in hlines:
                if l.startswith("ATOM") or l.startswith("HETATM"):
                    f.write(l[:21] + 'A' + l[22:])
                else:
                    f.write(l)

        # 2. Refine ONLY the Nanobody with PIPPack
        protein = vars(from_pdb_file(temp_h, mse_to_met=True, ignore_non_std=False))
        # Transform
        protein_transformed = transform_structure(protein, n_chi_bins, sc_d_mask_from_seq=True)
        batch = collate_fn([protein_transformed])

        # Inference
        sample_results = sample_epoch(models, batch, temperature=0.0, device=device, n_recycle=3)

        # Resampling
        resample_args = {"sample_temp": 0.1, "clash_overlap_tolerance": 0.4, "pro_tolerance_factor": 12, "max_iters": 50, "metropolis_temp": 0.000005}
        for i in range(batch.S.shape[0]):
            temp_protein = {
                "S": sample_results["S"][i],
                "X": sample_results["X"][i],
                "X_mask": sample_results["X_mask"][i],
                "BB_D": sample_results["BB_D"][i],
                "residue_index": sample_results["residue_index"][i],
                "residue_mask": sample_results["residue_mask"][i],
                "chi_logits": sample_results["chi_logits"][i],
                "chi_bin_offset": sample_results["chi_bin_offset"][i] if "chi_bin_offset" in sample_results else None,
            }
            pred_xyz = sample_results["final_X"][i]
            resample_xyz, _ = resample_loop(temp_protein, pred_xyz, **resample_args)
            sample_results["final_X"][i] = resample_xyz

        pdb_str = pdbs_from_prediction(sample_results)[0]
        refined_h = "/content/refined_H.pdb"
        with open(refined_h, 'w') as f:
            f.write(pdb_str)

        with open(refined_h, 'r') as f:
            print(f"[DEBUG] PIPPack generated PDB string length: {len(f.read())}")

        # 3. Merge Refined Nanobody with Original Antigen securely using Biopython
        output_filename = os.path.join(output_directory, base_name)

        refined_struct = parser.get_structure("refined", refined_h)
        print(f"[DEBUG] Refined struct chains before rename: {[c.id for c in refined_struct[0]]}")

        for model in refined_struct:
            for chain in model:
                chain.id = nb_id

        antigen_struct = parser.get_structure("antigen", temp_a)
        print(f"[DEBUG] Antigen struct chains: {[c.id for c in antigen_struct[0]]}")

        for model in antigen_struct:
            for chain in model:
                if chain.id not in [c.id for c in refined_struct[0]]:
                    refined_struct[0].add(chain)

        print(f"[DEBUG] Final Merged struct chains: {[c.id for c in refined_struct[0]]}")

        io_merge = PDBIO()
        io_merge.set_structure(refined_struct)
        io_merge.save(output_filename)

        print(f"Refined and merged {base_name} -> {output_filename}")

        for temp_file in [temp_h, temp_a, refined_h]:
            if os.path.exists(temp_file):
                try: os.remove(temp_file)
                except: pass
    print("\n✅ All designs refined.")

In [ ]:
#@title 5.5. Visualize refined designs with Side Chains and CDRs
import os, glob, re
import py3Dmol

output_directory = "/content/PIPPack_outputs" #@param {type:"string"}
design_index = 0 #@param {type:"integer"}

designs = sorted(glob.glob(os.path.join(output_directory, "*.pdb")))
if not designs:
    print("No designs found. Run the refinement cell first.")
else:
    print("Available refined designs:")
    for i, d in enumerate(designs):
        print(f" [{i}] {os.path.basename(d)}")

    idx = min(max(0, design_index), len(designs)-1)
    selected_design = designs[idx]
    pdb_data = open(selected_design).read()

    # Get CDR positions from the original FASTA
    fasta_path = "/content/IgGM_input.fasta"
    cdr1_range = cdr2_range = cdr3_range = None
    if os.path.exists(fasta_path):
        nb_seq = ""
        with open(fasta_path) as f:
            lines = f.read().splitlines()
            for i, l in enumerate(lines):
                if l.startswith(">H"):
                    nb_seq = lines[i+1]
                    break
        x_blocks = list(re.finditer(r'X+', nb_seq))
        if len(x_blocks) >= 3:
            # 1-based indexing for py3Dmol
            cdr1_range = [x_blocks[0].start() + 1, x_blocks[0].end()]
            cdr2_range = [x_blocks[1].start() + 1, x_blocks[1].end()]

            # For CDR3, determine length dynamically from the PDB file H chain
            tail_len = len(nb_seq) - x_blocks[2].end()
            h_atoms = [l for l in pdb_data.splitlines() if l.startswith('ATOM') and l[21] == 'H']
            if h_atoms:
                # find max residue index
                h_len = max(int(l[22:26].strip()) for l in h_atoms)
                cdr3_range = [x_blocks[2].start() + 1, h_len - tail_len]

    n_atoms = sum(1 for l in pdb_data.splitlines() if l.startswith('ATOM') or l.startswith('HETATM'))
    n_side = sum(1 for l in pdb_data.splitlines() if (l.startswith('ATOM') or l.startswith('HETATM')) and l[12:16].strip() not in ('N', 'CA', 'C', 'O', 'H'))
    print(f"\nShowing: {os.path.basename(selected_design)}")
    print(f"ATOM lines: {n_atoms} | side-chain atoms: {n_side} | all-atom: {n_side > 0}")

    view = py3Dmol.view(width=820, height=560)
    view.addModel(pdb_data, 'pdb')

    # Color Antigen (A) and Nanobody (H) backbone
    view.setStyle({'chain': 'A'}, {'cartoon': {'color': 'orange'}})
    view.setStyle({'chain': 'H'}, {'cartoon': {'color': 'white'}})

    # Color CDRs differently if found
    if cdr1_range:
        view.setStyle({'chain': 'H', 'resi': f"{cdr1_range[0]}-{cdr1_range[1]}"}, {'cartoon': {'color': 'red'}})
        view.setStyle({'chain': 'H', 'resi': f"{cdr2_range[0]}-{cdr2_range[1]}"}, {'cartoon': {'color': 'green'}})
        view.setStyle({'chain': 'H', 'resi': f"{cdr3_range[0]}-{cdr3_range[1]}"}, {'cartoon': {'color': 'blue'}})
        print(f"Colored CDRs: CDR1 (red) {cdr1_range}, CDR2 (green) {cdr2_range}, CDR3 (blue) {cdr3_range}")

    # Add transparent surfaces
    view.addSurface(py3Dmol.VDW, {'opacity': 0.4, 'color': 'orange'}, {'chain': 'A'})
    view.addSurface(py3Dmol.VDW, {'opacity': 0.4, 'color': 'white'}, {'chain': 'H'})

    # Highlight Nanobody and Antigen sidechains
    view.addStyle({'chain': 'H'}, {'stick': {'radius': 0.15, 'colorscheme': 'cyanCarbon'}})
    view.addStyle({'chain': 'A'}, {'stick': {'radius': 0.15, 'colorscheme': 'orangeCarbon'}})

    if cdr1_range:
        view.addStyle({'chain': 'H', 'resi': f"{cdr1_range[0]}-{cdr1_range[1]}"}, {'stick': {'radius': 0.2, 'colorscheme': 'redCarbon'}})
        view.addStyle({'chain': 'H', 'resi': f"{cdr2_range[0]}-{cdr2_range[1]}"}, {'stick': {'radius': 0.2, 'colorscheme': 'greenCarbon'}})
        view.addStyle({'chain': 'H', 'resi': f"{cdr3_range[0]}-{cdr3_range[1]}"}, {'stick': {'radius': 0.2, 'colorscheme': 'blueCarbon'}})

    view.zoomTo({'chain': 'H'})
    view.show()


In [ ]:
#@title 6. Download PIPPack Refined Designs (for AntiFold)
import os, glob, shutil
from google.colab import files
output_directory = "/content/PIPPack_outputs"

designs = sorted(glob.glob(os.path.join(output_directory, "*.pdb")))
if not designs:
    print("No designs to download. Run the refinement cell first.")
else:
    archive = shutil.make_archive("/content/Refined_designs", 'zip', output_directory)
    print(f"Zipped {len(designs)} refined design(s) -> {archive}")
    files.download(archive)
    print("\n➡️ Next step: Upload these to AntiFold.")


---
### ✅ Step complete — disconnect before the next notebook

To free the GPU/CPU for the next step, **disconnect and delete this runtime**:
`Runtime → Disconnect and delete runtime` (or `Manage sessions → Terminate`).

⬅️ **[Back to the main workflow](https://biochorl.github.io/Nanobody_de_novo_design/)**
